# M3 outputs: risk and budget plan

Loads every file M3 writes to `outputs/` (parquet + JSON) and shows them cell by cell.

**Before running:** pick the project's `.venv` as the kernel (it has pandas + pyarrow). To refresh the files first, run `python -m pipeline.risk --no-llm` from the repo root.

All data is synthetic.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# find the repo root (works whether the notebook runs from notebooks/ or the root)
ROOT = Path.cwd()
while not (ROOT / "outputs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "outputs"
print("outputs folder:", OUT)
sorted(p.name for p in OUT.iterdir())

## 1. Run report (`risk_report.json`)
Weights used, the as-of date, where open-issue demand came from, and plan totals.

In [ ]:
report = json.loads((OUT / "risk_report.json").read_text(encoding="utf-8"))
print(json.dumps(report, indent=1, ensure_ascii=False))

## 2. Road risk (`road_risk.parquet`)
One row per road segment (291). `risk_score` = 100 × Σ weight × component.

In [ ]:
road = pd.read_parquet(OUT / "road_risk.parquet")
print(road.shape)
road.head()

In [ ]:
# top 15 riskiest roads in the city
cols = ["road_id", "ward_code", "road_name", "risk_score", "c_exposure", "c_safety",
        "c_critical", "c_flood", "c_condition", "open_issues", "under_dlp", "reason"]
road.sort_values("risk_score", ascending=False)[cols].head(15)

In [ ]:
# component and score distribution
road[["risk_score", "c_exposure", "c_safety", "c_critical", "c_flood", "c_condition"]].describe().round(3)

In [ ]:
# check: the score rebuilds exactly from the weights in the report
w = report["weights"]
rebuilt = 100 * sum(w[c] * road[c] for c in w)
print("max difference:", round((rebuilt - road.risk_score).abs().max(), 4))

## 3. Ward risk (`ward_risk.parquet`)
One row per ward (24). Negative `remaining_budget_inr` = ward already over budget.

In [ ]:
ward = pd.read_parquet(OUT / "ward_risk.parquet")
print(ward.shape)
ward.sort_values("risk_score", ascending=False)

In [ ]:
# wards already over budget (they get no new paid repairs)
ward[ward.remaining_budget_inr <= 0][["ward_code", "monsoon_repair_budget_inr_2025", "billed_inr", "remaining_budget_inr"]]

## 4. Method durability (`method_durability.parquet`)
Re-open rate per repair method. `recommended` = most durable method for that work type.

In [ ]:
methods = pd.read_parquet(OUT / "method_durability.parquet")
methods[["work_type", "repair_method", "n_jobs", "reopen_rate_raw", "reopen_rate",
         "est_cost_inr", "cost_per_lasting_repair_inr", "method_rank", "recommended"]]

## 5. Budget plan (`budget_plan.parquet`)
Ranked plan per ward: `Repair`, `Invoke DLP` (₹0 to BMC) or `Monitor`.

In [ ]:
plan = pd.read_parquet(OUT / "budget_plan.parquet")
print(plan.shape)
plan.action.value_counts()

In [ ]:
# per-ward summary: repairs, warranty claims, spend vs money left
summary = plan.groupby("ward_code").agg(
    repairs=("action", lambda s: (s == "Repair").sum()),
    invoke_dlp=("action", lambda s: (s == "Invoke DLP").sum()),
    monitor=("action", lambda s: (s == "Monitor").sum()),
    planned_spend_inr=("est_cost_inr", "sum"),
    remaining_budget_inr=("remaining_budget_inr", "first"),
)
summary["unspent_inr"] = summary.remaining_budget_inr.clip(lower=0) - summary.planned_spend_inr
summary

In [ ]:
# look at one ward's plan: change WARD to any ward code (A, B, F/S, L, ...)
WARD = "F/S"
plan[plan.ward_code == WARD][["rank", "road_id", "action", "recommended_method", "est_cost_inr",
                              "cumulative_inr", "remaining_budget_inr", "risk_score", "reason"]]

In [ ]:
# checks: no ward overspends, warranty roads cost nothing
spend = plan[plan.action == "Repair"].groupby("ward_code").est_cost_inr.sum()
left = plan.groupby("ward_code").remaining_budget_inr.first().clip(lower=0)
print("wards over remaining budget:", int((spend - left.reindex(spend.index) > 1).sum()))
print("DLP rows with cost > 0:", int((plan.loc[plan.action == "Invoke DLP", "est_cost_inr"] > 0).sum()))

## 6. Ward memos (`ward_memos.json`)
Plain-language budget memo per ward (template now; Claude once M1's wrapper exists).

In [ ]:
memos = json.loads((OUT / "ward_memos.json").read_text(encoding="utf-8"))
memo_df = pd.DataFrame([{"ward_code": k, "source": v["source"], "memo": v["memo"]} for k, v in memos.items()])
memo_df

In [ ]:
# full memo + the facts it was built from, for one ward
print(memos[WARD]["memo"])
print()
print(json.dumps(memos[WARD]["facts"], indent=1, ensure_ascii=False))